# ask_only

Ask a finished run's saved checkpoints report questions again. **Nothing is trained**, and the 1,000 decision trials
are not asked again.

**To use it:** edit the Settings cell, then Run All. That is the whole notebook: every cell below the Settings cell
does only what the Settings cell says.

## What Run All does

1. Sets up the machine: clones or updates the code (`BRANCH`), installs it, asks for the Hugging Face token.
2. Prints the plan: every run in `RUN`, its steps and how many replies it will ask. Every setting is checked here,
   before a model is loaded, so a mistyped run id or question name stops in seconds.
3. For each run in `RUN`, one after another: downloads that run's adapters and saved decisions from `HUB_REPO` if
   they are not on this machine, loads the base model, applies each step's adapter, asks the questions, pushes the
   output folder to `HUB_REPO`, and drops the model.
4. Shows one table per run, and pushes every output folder once more.

It can be stopped and started again: a run whose output folder already lists every step (here or on the Hub) is
skipped. A run that fails does not stop the runs after it; the last cell names it.

## What comes out

One folder per run, `results/<run_id>/ask_only/<NAME>/`, pushed to the Hub at the same path. The run's own files
are never changed.

| file | what |
|---|---|
| `settings.json` | the settings the questions were asked under |
| `reports_raw_decision_step-<N>.jsonl` | every reply, with the question, the prompt, the trial's kind and both options' values |
| `step_answers.md` / `.csv` | every question's answers per step |
| `faithfulness_by_trial_kind.csv` | the weights cosine per trial kind (`cos_recovered_4` is the one to quote) |
| `cutoff_flags.csv` / `_summary.csv` | per cut reply: dimension right, distance from the cut, by trial kind |
| `reply_forms.csv` | weight replies per trial kind, by form |
| `timing.csv` | seconds per step; written last, so a folder that has it is complete |

Replies are scored against the rule refitted from the decisions the run saved at that step
(`selections_decision_step-<N>.csv`). A step the run did not score cannot be asked, and neither can step 0 for a run
that did not score the base model.

Code: `src/ask_only.py` (`ask_runs`). Tests: `python -m tests.test_ask_only` and
`python -m tests.test_ask_only_notebook`, which runs every cell of this notebook with a stand-in for the model.

## Settings

Edit this cell only.

| setting | what it means |
|---|---|
| `RUN` | The run to ask: its id, or just its timestamp. A list asks each run in turn with the same settings. |
| `STEPS` | The saved checkpoints to ask, by training step. |
| `QUESTIONS_FROM` | A settings file to take the questions from. `None` = each run's own questions. |
| `ONLY` | A list of question names, to ask only those. `None` = every question. |
| `SHOW_NEITHER` | `True` = the trial lists option C, "Neither option", as the decision task does. |
| `REPLY_PREFIX` | The prefill: text the reply is made to start with. `"{"` starts it as JSON. `""` = none. |
| `PER_KIND` | How many tea pairs each question is asked on. See below. |
| `FLIP_AB` | `True` = every pair shown with A's and B's values swapped (the same pairs). |
| `EVAL_BATCH_SIZE` | Prompts sent through the model at once. `None` = the size that fits this machine's card. It changes the speed, not the replies. |
| `NAME` | The output folder's name. One name holds one set-up: asking the same name with other settings is refused. |
| `HUB_REPO` | Where missing files are downloaded from and the output is saved to. `None` = neither. |
| `BRANCH` | The branch of the code to run. |

**`PER_KIND`.** A question that shows a tea pair is asked once per pair. Pairs come in three kinds, by the
persona's cut: both teas pass it (the weights decide), one passes and one fails (the cut decides), both fail (the
answer is C). `PER_KIND = 50` draws pairs until there are 50 of each kind, 150 pairs, so each kind has enough
replies to be read on its own. `PER_KIND = None` asks the run's own 50 pairs, whatever mix of kinds they are. The
run's own 50 are always among the 150: they are the rows with `trial_draw` below 50 in `reports_raw`.

**As it ships** the cell asks the listing-order control on A seed 5 and A-inverse seed 5: the nine questions with
the attributes listed 5, 3, 4, 1, 2 instead of 1 to 5, each attribute under its own number, so the replies are scored
as the nine's are (`configs/ask_only/nine_q_order53412.json`, `scripts/make_listing_order_control.py`). It reads
beside the same runs' `nine-q-prefix-c-perkind50` folder, asked the same way with the attributes in order.

The cell before it asked the eight nine-question seed-repeat runs with the prefill and C on: `RUN` = 20261123-180745,
20261124-022059, 20261123-180741, 20261124-170211, 20261124-011704, 20261124-003233, 20261123-234256,
20261123-131618, `QUESTIONS_FROM = None`, `NAME = "nine-q-prefix-c-perkind50"`. For those runs with A and B swapped,
set `FLIP_AB = True` and `NAME = "nine-q-prefix-c-perkind50-flipped"`.

In [ ]:
RUN = ["20261123-180745",          # a-nine-q-seed5
       "20261123-180741"]          # a-inverse-nine-q-seed5
STEPS = [250, 500, 750, 1000, 1250, 1500, 1750, 2000, 2250, 2500, 2750, 3000]   # saved decision steps
QUESTIONS_FROM = "configs/ask_only/nine_q_order53412.json"   # the nine, attributes listed 5, 3, 4, 1, 2; None = each run's own questions
ONLY = None                        # a list of question names; None = every question
SHOW_NEITHER = True                # True = option C ("Neither option") is shown
REPLY_PREFIX = "{"                 # the prefill; "" = none
PER_KIND = 50                      # tea pairs per question: 50 of each kind (150); None = the run's own 50 pairs
FLIP_AB = False                    # True = every pair shown with A's and B's values swapped
EVAL_BATCH_SIZE = None             # None = the size that fits this machine's card (192 on 80 GB or more); or a number
NAME = "nine-q-order53412-prefix-c-perkind50"   # output folder: results/<run_id>/ask_only/<NAME>/
HUB_REPO = "adamchanna/bdp-plunkett-qwen3"   # missing files are downloaded from it and the output is saved to it; None = neither
BRANCH = "bootstrap-ci"            # the branch of the code to run

## Set up the machine

The same cells as `run_experiment.ipynb`, so this notebook runs on its own. On a fresh machine they clone the repo
(`BRANCH`) and install what it needs; on a machine that is already set up they bring the code up to date. The last
one asks for the Hugging Face token the save cells need.

In [ ]:
!nvidia-smi || echo "no nvidia-smi: CPU-only machine, an 8B run needs a GPU"

In [ ]:
# Clone BRANCH when the code is not already here, or move an existing clone onto it, then pin versions.
# run_me.py runs before the install: it re-pins requirements to whatever this machine
# already ships, keeping its CUDA-matched torch instead of letting pip swap it out.
import os, subprocess, sys

CLONE_URL = "https://github.com/Bilal-Trigui/Decision_Task_Database_Experiments.git"
if not os.path.exists("src"):
    if not os.path.exists("Decision_Task_Database_Experiments"):
        subprocess.run(["git", "clone", "-b", BRANCH, CLONE_URL], check=True)
    os.chdir("Decision_Task_Database_Experiments")
# An existing clone may sit on another branch or an older commit. This refuses rather than
# overwrite local edits: commit or stash them, or set BRANCH to the branch they are on.
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(f"code: branch {BRANCH}, commit {commit}")
subprocess.run([sys.executable, "run_me.py"], check=True)

# Keep model downloads in one folder, easy to delete on a shared disk.
os.environ.setdefault("HF_HOME", os.path.abspath("hf_cache"))
# Let the CUDA allocator grow segments instead of fragmenting. Must be set before
# torch initialises CUDA, which is why it lives here and not in the GPU check.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("repo ready in", os.getcwd())

In [ ]:
# %pip installs into THIS kernel's environment. A pip subprocess can land packages
# somewhere the kernel cannot import from, which surfaces later as ModuleNotFoundError.
%pip install -q -r requirements.txt

In [ ]:
# Confirm the kernel can import what the pipeline needs.
#
# On JupyterHub, pip usually installs into the user site-packages, and a kernel
# that started before that folder existed does not have it on sys.path. That is
# the normal cause of ModuleNotFoundError straight after a successful install.
# Adding the folder here repairs it without a restart.
import importlib, importlib.util, os, site, sys

user_site = site.getusersitepackages()
if user_site and os.path.isdir(user_site) and user_site not in sys.path:
    sys.path.append(user_site)
    importlib.invalidate_caches()
    print("added user site-packages to sys.path:", user_site)

print("kernel python:", sys.executable)
needed = ("numpy", "pandas", "sklearn", "matplotlib", "torch", "transformers", "peft", "huggingface_hub")
missing = [m for m in needed if importlib.util.find_spec(m) is None]
if missing:
    print("MISSING:", missing)
    print("Restart the kernel (Kernel -> Restart Kernel), run the install cell, then this cell again.")
else:
    print("all imports available")

In [ ]:
# Confirm the GPU. The precision is the run's own setting (bf16 needs Ampere or newer); the batch size is
# EVAL_BATCH_SIZE, or the size that fits this card when that is None.
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("vram GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 supported", torch.cuda.is_bf16_supported())
else:
    print("no CUDA device: an 8B run needs a GPU")

In [ ]:
# Credentials. Typed rather than written into a cell, so nothing lands in git or in
# a saved notebook. Needed only to save to the Hub: downloading from a public repo and
# the public Qwen3 weights do not require a token.
#
# A token that the Hub rejects is discarded and you are asked again, so a typo or
# a revoked token never gets stuck in the environment.
import getpass, os
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

who = None
for attempt in range(3 if HUB_REPO else 0):
    token = os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face WRITE token (blank to skip): ")
    if not token:
        break
    try:
        who = HfApi(token=token).whoami()
        os.environ["HF_TOKEN"] = token
        break
    except HfHubHTTPError as err:
        os.environ.pop("HF_TOKEN", None)
        if err.response is not None and err.response.status_code == 401:
            print("The Hub rejected that token (401). Check it is a WRITE token and not revoked. Try again.")
            continue
        raise

if not HUB_REPO:
    print("HUB_REPO is None: nothing is downloaded from the Hub and nothing is saved to it")
elif who is None:
    print("no token: the notebook works, but saving refuses and nothing survives the machine")
else:
    owner = HUB_REPO.split("/")[0]
    allowed = [who["name"]] + [o["name"] for o in who.get("orgs", [])]
    print("token belongs to:", who["name"])
    if owner in allowed:
        print(f"HUB_REPO owner '{owner}' matches. Saving will work.")
    else:
        print(f"MISMATCH: HUB_REPO starts with '{owner}' but this token can only write to {allowed}.")
        print("Saving will be refused. Use a token of that account, or name a Hub repo you can write to in HUB_REPO.")

## Imports

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "ask_only.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("run this notebook from inside the project folder")
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Forget any copy of the project's code this session imported earlier, so the code the set-up cell just pulled is
# the code that runs. Without this a session goes on with the old code until it is restarted.
for module in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[module]

import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)

from src import ask_only as A
from src.persist import push_folder
print("project:", ROOT)

## Ask

Prints the plan, then asks each run in `RUN`. Files a run needs are downloaded from `HUB_REPO` just before it is
asked, and its output folder is pushed as soon as it is written, so stopping part-way loses only the run that was
being asked. Run this cell again to carry on: finished runs are skipped.

If the card runs out of memory the cell stops and names a smaller `EVAL_BATCH_SIZE` to set.

In [ ]:
JOBS = A.ask_runs(RUN, STEPS, QUESTIONS_FROM, ONLY, SHOW_NEITHER, REPLY_PREFIX, PER_KIND, NAME, flip_ab=FLIP_AB,
                  eval_batch_size=EVAL_BATCH_SIZE, hub_repo=HUB_REPO)
pd.DataFrame(JOBS).drop(columns="out_dir")

## Results

One table per run: a row per step. A run that was skipped because the Hub already has it shows no table here.

In [ ]:
for job in JOBS:
    print(f"== {job['run_id']}: {job['status']}")
    if (job["out_dir"] / "step_answers.csv").exists():
        try:
            display(A.report_table(job["out_dir"]))
        except Exception as err:      # a table is never worth stopping the save below for
            print(f"   no table: {type(err).__name__}: {err}")

## Save it off the machine

The Ask cell has already pushed each folder it wrote. This cell pushes every `results/<run_id>/ask_only/` folder on
this machine again, so nothing is left behind: run it before you leave a rented machine. It stops with the reason
when the Hub refuses. Last, it stops with an error naming any run the Ask cell could not ask.

In [ ]:
folders = sorted((ROOT / "results").glob("*/ask_only"))
for folder in folders:
    push_folder(folder, HUB_REPO)
print(f"{len(folders)} ask_only folder(s) on this machine" + ("" if HUB_REPO else "; HUB_REPO is None, so none was saved"))
A.stop_if_failed(JOBS)

## Checking the tool against a run (not part of Run All)

Asked the way a run was scored, the replies should match the run's own saved replies. To check: set `RUN` to one
run and `STEPS` to one step, `QUESTIONS_FROM = None`, `ONLY = None`, `PER_KIND = None`, `SHOW_NEITHER` and
`REPLY_PREFIX` to what the run used (its `results/<run_id>/settings.json`), and a new `NAME`. Run All, then in a new
cell:

```python
from src import download as DL
run_id = DL.fetch_run(HUB_REPO, RUN, STEPS, results=("reports_raw",))
A.reproduction(A.find_run(run_id), JOBS[0]["out_dir"], STEPS[0])[0]
```

`match_rate` should be near 1 for every question.